# brick3_instruct_control — Colab

**Goal.** Clean-sheet control: Qwen2.5-0.5B-**Instruct** + `drill_langues_full` + LoRA r=16, no warm-start.

**Delta vs brick3_local.** Only the base changes (Instruct vs Base). All other variables held constant (500 steps, r=16, lr=1e-4, same data).

**Hypothesis.** The polly-1 smoke test showed the 0.5B **Base** model learns tongue tagging but fails structural compliance (prefix_match=0%). If the Instruct prior cracks it, we know the bottleneck is the base variant, not the data. If it doesn't, the bottleneck is data shape.

**Local launcher reference:** `scripts/train/brick3_instruct_control.py`.

**Runtime:** T4 GPU is enough. ~45–60 min for 500 steps.

## 1. Install dependencies

In [ ]:
%pip -q install -U "transformers>=4.45" "trl>=0.12" "peft>=0.13" "datasets>=2.20" "accelerate>=0.34" "bitsandbytes>=0.44" huggingface_hub

## 2. Auth + clone repo

The repo is public on the `feature/cli-code-tongues` branch. If you need HF push at the end, set `HF_TOKEN` from Colab secrets.

In [ ]:
import os
from google.colab import userdata

try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN loaded')
except Exception as e:
    print(f'no HF_TOKEN in secrets: {e} (HF push at the end will be skipped)')

!git clone --depth 1 -b feature/cli-code-tongues https://github.com/issdandavis/SCBE-AETHERMOORE.git /content/scbe
%cd /content/scbe
!git log -1 --oneline

## 3. Verify training data is present

In [ ]:
!wc -l training-data/sft/drill_langues_full_train.sft.jsonl training-data/sft/drill_langues_full_holdout.sft.jsonl

Expected:
- train: 2373 rows
- holdout: 257 rows

## 4. Run the training script

This is the unmodified local launcher. All knobs are baked in: 500 steps, r=16, lr=1e-4, BATCH=4 × GRAD_ACCUM=4, cosine schedule with 25-step warmup, eval every 50, save every 100.

In [ ]:
!python scripts/train/brick3_instruct_control.py

## 5. Inspect final eval metrics

In [ ]:
import json

state_path = 'artifacts/training/brick3_instruct_control/trainer_state_final.json'
state = json.loads(open(state_path).read())

print(f"base:        {state['base']}")
print(f"train rows:  {state['train_rows']}")
print(f"eval rows:   {state['eval_rows']}")
print(f"max steps:   {state['max_steps']}")
print(f"lora rank:   {state['lora_r']}")
print(f"lr:          {state['lr']}")

eval_entries = [e for e in state['log_history'] if 'eval_loss' in e]
if eval_entries:
    first, last = eval_entries[0], eval_entries[-1]
    print(f"\nfirst eval:  step={first.get('step')} loss={first['eval_loss']:.4f}")
    print(f"last eval:   step={last.get('step')} loss={last['eval_loss']:.4f}")

    best = min(eval_entries, key=lambda e: e['eval_loss'])
    print(f"best eval:   step={best.get('step')} loss={best['eval_loss']:.4f}")

## 6. Save adapter to Drive (optional)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
src = '/content/scbe/artifacts/training/brick3_instruct_control'
dst = '/content/drive/MyDrive/scbe/artifacts/training/brick3_instruct_control'
shutil.copytree(src, dst, dirs_exist_ok=True)
print(f'saved to: {dst}')

## 7. Push adapter to Hugging Face (optional)

Only runs if `HF_TOKEN` was loaded in step 2.

In [ ]:
import os
from pathlib import Path
from huggingface_hub import HfApi, create_repo

if not os.environ.get('HF_TOKEN'):
    print('no HF_TOKEN — skipping push')
else:
    repo_id = 'issdandavis/brick3-instruct-control-qwen-0p5b'
    final_dir = Path('/content/scbe/artifacts/training/brick3_instruct_control/final')
    if not final_dir.exists():
        print(f'no final adapter at {final_dir} — training may have failed')
    else:
        api = HfApi()
        create_repo(repo_id, exist_ok=True, private=True, token=os.environ['HF_TOKEN'])
        api.upload_folder(
            folder_path=str(final_dir),
            repo_id=repo_id,
            token=os.environ['HF_TOKEN'],
            commit_message='brick3_instruct_control initial push',
        )
        print(f'pushed: https://huggingface.co/{repo_id}')